# 03 — Repeated classification split verification

This notebook creates or loads the checksummed patient split registry and verifies balance, disjointness, nesting, tube coverage, and classifier fairness. It does **not** train classifiers.

In [ ]:
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from flowlot.evaluation.repeated_benchmark import (
    audit_registry, create_job_table, create_split_registry,
    export_legacy_splits_h5, load_registry,
)
from flowlot.io import audit_stage2

In [ ]:
STAGE2 = Path('../data/stage2_analytics.h5')
DATASET, CELL_COUNT = 'BLAST110', '1000'
TUBES = ['P1', 'P2', 'P3', 'P4']
RESULTS = Path('../results/repeated_classification')
TRAIN_PER_CLASS = (2, 4, 6, 8)
REPEATS, TEST_SIZE, SEED = 10, 0.5, 42
CREATE_SPLITS = False
RESULTS.mkdir(parents=True, exist_ok=True)

## Verify cohort availability before splitting

Strict across-model comparisons use the patient intersection so every single-tube and cell model receives valid data for the same IDs.

In [ ]:
inventories, stage2_issues = audit_stage2(STAGE2)
assert stage2_issues.empty, 'Stage 2 audit failed before splitting'
raw = inventories['raw'].query('dataset == @DATASET and cell_count == @CELL_COUNT and tube in @TUBES')
coverage = raw.assign(present=1).pivot_table(
    index=['patient_id', 'label'], columns='tube', values='present', aggfunc='max', fill_value=0
)
coverage['available_tubes'] = coverage.sum(axis=1)
display(coverage.groupby(['label', 'available_tubes']).size().rename('patients').reset_index())
complete_ids = coverage.index.get_level_values('patient_id')[coverage['available_tubes'] == len(TUBES)]
print(f'{len(complete_ids)} patients have all {len(TUBES)} requested tubes')

## Create or load the immutable registry

The legacy HDF5 export contains the same IDs as the JSON registry. The checksum detects any later manual editing.

In [ ]:
registry_path = RESULTS / 'shared_splits.json'
if CREATE_SPLITS:
    registry = create_split_registry(
        STAGE2, DATASET, CELL_COUNT, registry_path, TRAIN_PER_CLASS, REPEATS,
        TEST_SIZE, SEED, tubes=TUBES, patient_policy='intersection',
    )
    export_legacy_splits_h5(registry, RESULTS / 'legacy_splits.h5')
else:
    registry = load_registry(registry_path)
print('Registry SHA-256:', registry['registry_hash'])

## Balance, overlap, and nesting assertions

Within a repeat, all k values use one fixed test cohort and `k=2 ⊂ 4 ⊂ 6 ⊂ 8`. Every training cohort has exactly k patients from every class.

In [ ]:
audit = pd.DataFrame(audit_registry(registry))
display(audit)
assert (audit.groupby('run')['test_ids_hash'].nunique() == 1).all()
for split in registry['splits']:
    test = set(split['test_ids'])
    cohorts = [set(split['train_ids_by_k'][str(k)]) for k in TRAIN_PER_CLASS]
    assert all(not (cohort & test) for cohort in cohorts)
    assert all(left < right for left, right in zip(cohorts, cohorts[1:]))
print('PASS: balanced, nested, disjoint, and fixed-test invariants')

In [ ]:
labels = registry['labels']
class_names = registry['class_names']
rows, test_frequency = [], Counter()
for split in registry['splits']:
    test_frequency.update(split['test_ids'])
    for k, ids in split['train_ids_by_k'].items():
        counts = Counter(labels[patient] for patient in ids)
        rows.extend({'run': split['run'], 'k': int(k), 'class': class_names[label], 'patients': count} for label, count in counts.items())
class_balance = pd.DataFrame(rows)
display(class_balance.pivot_table(index=['run', 'k'], columns='class', values='patients'))
frequency = pd.DataFrame({'patient_id': test_frequency.keys(), 'test_appearances': test_frequency.values()})
display(frequency['test_appearances'].describe())
sns.histplot(frequency, x='test_appearances', discrete=True)
plt.title('Patient frequency across repeated test cohorts')
plt.show()

## Generate jobs and prove classifier fairness

The job table never stores independent samples. Every classifier row resolves its IDs from this one registry, and the accompanying audit CSV records their hashes.

In [ ]:
MODELS = ['logistic', 'nsc', 'flowsom', 'cellcnn', 'attention_mil', 'cytoset', 'dgcnn', 'pointnet2']
jobs = create_job_table(registry_path, RESULTS / 'jobs.tsv', MODELS, ('single', 'early_mean', 'late_soft'))
job_frame = pd.DataFrame(jobs)
display(job_frame.groupby(['model', 'aggregation', 'tube']).size().rename('jobs').reset_index())
print(f'PASS: {len(jobs):,} jobs reference registry {registry["registry_hash"][:16]}…')